## Example 1: marshmallow functionality test 1

In [ ]:
from marshmallow import Schema, fields, post_load
from datetime import datetime

# Example 1: Basic schema definition and object serialization
print("Example 1: Basic Schema Definition and Object Serialization")

try:
    # Define a user data class
    class User:
        def __init__(self, name, email, created_at=None):
            self.name = name
            self.email = email
            self.created_at = created_at or datetime.now()
        
        def __repr__(self):
            return f"<User(name='{self.name}', email='{self.email}')>"
    
    # Define schema for User serialization
    class UserSchema(Schema):
        name = fields.Str(required=True)
        email = fields.Email(required=True)
        created_at = fields.DateTime(format='%Y-%m-%d %H:%M:%S')
        
        @post_load
        def make_user(self, data, **kwargs):
            return User(**data)
    
    print("Schema defined for User object")
    
    # Create sample users
    users = [
        User("Alice Johnson", "alice@example.com"),
        User("Bob Smith", "bob@example.com"),
        User("Carol Davis", "carol@company.org")
    ]
    
    print(f"Created {len(users)} user objects")
    
    # Serialize users to dictionaries
    user_schema = UserSchema()
    
    print("\nSerialization (objects to dicts):")
    serialized_users = []
    
    for user in users:
        user_dict = user_schema.dump(user)
        serialized_users.append(user_dict)
        print(f"  {user.name}: {user_dict}")
    
    # Serialize multiple users at once
    users_schema = UserSchema(many=True)
    all_users_data = users_schema.dump(users)
    
    print(f"\nBulk serialization: {len(all_users_data)} users serialized")
    
    # Deserialize back to objects
    print(f"\nDeserialization (dicts to objects):")
    
    for user_data in serialized_users[:2]:  # Test first 2
        try:
            user_obj = user_schema.load(user_data)
            print(f"  Loaded: {user_obj}")
        except Exception as e:
            print(f"  Load error: {e}")
            raise
    
    # Show schema capabilities
    print(f"\nSchema capabilities:")
    capabilities = [
        "Field validation: Required fields, type checking, format validation",
        "Object conversion: Automatic conversion between objects and dicts",
        "Bulk operations: Handle lists of objects with many=True",
        "Custom serialization: Control how data is serialized/deserialized",
        "Error handling: Comprehensive validation error reporting"
    ]
    
    for capability in capabilities:
        print(f"  • {capability}")
    
    print("Basic schema and serialization completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    # Fallback demonstration
    print("Demonstrating serialization concepts:")
    concepts = [
        "Schema definition: Declarative field specification for objects",
        "Object serialization: Convert Python objects to primitive types",
        "Data validation: Ensure data meets schema requirements", 
        "Type conversion: Automatic conversion between data types",
        "Bidirectional mapping: Serialize and deserialize with same schema"
    ]
    
    for concept in concepts:
        print(f"  - {concept}")
    raise  # re-raise: a broken package must fail this example, not print past it

## Example 2: marshmallow functionality test 2

In [ ]:
from marshmallow import Schema, fields, validate, ValidationError
from decimal import Decimal

# Example 2: Advanced validation and custom fields
print("Example 2: Advanced Validation and Custom Fields")

try:
    # Define product schema with advanced validation
    class ProductSchema(Schema):
        name = fields.Str(
            required=True,
            validate=validate.Length(min=1, max=100)
        )
        price = fields.Decimal(
            required=True,
            validate=validate.Range(min=0, min_inclusive=False),
            places=2
        )
        category = fields.Str(
            required=True,
            validate=validate.OneOf(['electronics', 'books', 'clothing', 'home'])
        )
        tags = fields.List(
            fields.Str(),
            validate=validate.Length(min=1, max=10)
        )
        sku = fields.Str(
            required=True,
            validate=validate.Regexp(r'^[A-Z]{3}-\d{4}$')
        )
        in_stock = fields.Bool(load_default=True)  # `missing=` was removed in marshmallow 4
        rating = fields.Float(validate=validate.Range(min=0, max=5))
    
    print("Advanced product schema with validation created")
    
    # Test valid products
    valid_products = [
        {
            'name': 'Wireless Headphones',
            'price': '99.99',
            'category': 'electronics',
            'tags': ['wireless', 'bluetooth', 'audio'],
            'sku': 'ELE-1001',
            'rating': 4.5
        },
        {
            'name': 'Python Programming Book',
            'price': '45.00',
            'category': 'books',
            'tags': ['programming', 'python', 'education'],
            'sku': 'BOK-2001',
            'rating': 4.8
        }
    ]
    
    print(f"\nValidating {len(valid_products)} valid products:")
    
    schema = ProductSchema()
    validated_products = []
    
    for product in valid_products:
        try:
            result = schema.load(product)
            validated_products.append(result)
            print(f"  ✓ {product['name']}: Valid")
        except ValidationError as err:
            print(f"  ✗ {product['name']}: {err.messages}")
            raise AssertionError(f"schema.load rejected a valid product: {product['name']}") from err
    
    # Test invalid products
    invalid_products = [
        {
            'name': '',  # Too short
            'price': '99.99',
            'category': 'electronics',
            'sku': 'ELE-1001'
        },
        {
            'name': 'Bad Price Product',
            'price': '-10.00',  # Negative price
            'category': 'electronics',
            'sku': 'ELE-1001'
        },
        {
            'name': 'Wrong Category',
            'price': '25.00',
            'category': 'invalid_category',  # Not in OneOf
            'sku': 'ELE-1001'
        },
        {
            'name': 'Bad SKU Format',
            'price': '15.00',
            'category': 'books',
            'sku': 'invalid-sku'  # Wrong format
        }
    ]
    
    print(f"\nTesting {len(invalid_products)} invalid products:")
    
    # each invalid product breaks exactly one rule: load() must reject it, naming that one field
    for product, bad_field in zip(invalid_products, ['name', 'price', 'category', 'sku']):
        try:
            result = schema.load(product)
        except ValidationError as err:
            error_fields = list(err.messages.keys())
            assert error_fields == [bad_field], err.messages
            print(f"  ✓ Correctly rejected: {error_fields}")
        else:
            raise AssertionError(f"schema.load accepted an invalid product: {product.get('name', 'unnamed')}")
    
    # Partial validation (only specific fields)
    print(f"\nPartial validation example:")
    
    partial_data = {'name': 'Partial Product', 'price': '29.99'}
    
    # Only validate provided fields: partial=True skips the missing required category/sku but still validates and
    # converts what is given; load_default is NOT applied to a skipped field, so no in_stock
    result = schema.load(partial_data, partial=True)
    assert result == {'name': 'Partial Product', 'price': Decimal('29.99')}, result
    print(f"  Partial validation passed: {list(partial_data.keys())}")
    
    # Custom field validation
    print(f"\nValidation features:")
    validation_features = [
        "Length validation: Min/max constraints on strings and lists",
        "Range validation: Numeric bounds with inclusive/exclusive options",
        "Choice validation: Restrict values to predefined options",
        "Pattern validation: Regular expression matching",
        "Type validation: Automatic type checking and conversion",
        "Custom validators: Define domain-specific validation rules"
    ]
    
    for feature in validation_features:
        print(f"  • {feature}")
    
    print("Advanced validation completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    # Fallback demonstration
    print("Demonstrating validation concepts:")
    concepts = [
        "Field validators: Built-in validation for common constraints",
        "Error reporting: Detailed error messages for failed validation",
        "Partial validation: Validate only provided fields",
        "Custom constraints: Define business-specific validation rules",
        "Type coercion: Automatic conversion with validation"
    ]
    
    for concept in concepts:
        print(f"  - {concept}")
    raise  # re-raise: a broken package must fail this example, not print past it

## Example 3: marshmallow functionality test 3

In [ ]:
from marshmallow import Schema, fields, pre_load, post_load, pre_dump, post_dump
from datetime import datetime, timedelta

# Example 3: Data transformation and preprocessing hooks
print("Example 3: Data Transformation and Preprocessing Hooks")

try:
    # Define order schema with data transformation
    class OrderSchema(Schema):
        order_id = fields.Str(required=True)
        customer_email = fields.Email(required=True)
        items = fields.List(fields.Dict(), required=True)
        total_amount = fields.Decimal(places=2)
        order_date = fields.DateTime()
        status = fields.Str(validate=validate.OneOf(['pending', 'processing', 'shipped', 'delivered']))
        
        # Preprocessing hooks - modify data before validation
        @pre_load
        def preprocess_input(self, data, **kwargs):
            # Normalize email to lowercase
            if 'customer_email' in data:
                data['customer_email'] = data['customer_email'].lower()
            
            # Add order_date if missing
            if 'order_date' not in data:
                data['order_date'] = datetime.now().isoformat()
            
            # Convert string amounts to proper format
            if 'total_amount' in data and isinstance(data['total_amount'], str):
                # Remove currency symbols and whitespace
                data['total_amount'] = data['total_amount'].replace('$', '').replace(',', '').strip()
            
            return data
        
        @post_load
        def create_order_summary(self, data, **kwargs):
            # Add computed fields after loading
            data['item_count'] = len(data.get('items', []))
            data['is_high_value'] = float(data.get('total_amount', 0)) > 100
            return data
        
        # Postprocessing hooks - modify output during serialization
        @pre_dump
        def prepare_for_output(self, data, **kwargs):
            # Convert datetime objects if needed
            if isinstance(data.get('order_date'), datetime):
                # Keep as is - marshmallow will handle datetime formatting
                pass
            return data
        
        @post_dump
        def format_output(self, data, **kwargs):
            # Add display formatting
            if 'total_amount' in data:
                data['formatted_total'] = f"${data['total_amount']}"
            
            # Add metadata
            data['processed_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            
            return data
    
    print("Order schema with transformation hooks created")
    
    # Test data with various formats
    raw_orders = [
        {
            'order_id': 'ORD-001',
            'customer_email': 'JOHN.DOE@EXAMPLE.COM',  # Will be normalized
            'items': [
                {'product': 'Laptop', 'price': 899.99, 'qty': 1},
                {'product': 'Mouse', 'price': 29.99, 'qty': 2}
            ],
            'total_amount': '$959.97',  # Will be cleaned
            'status': 'pending'
            # order_date will be added automatically
        },
        {
            'order_id': 'ORD-002', 
            'customer_email': 'jane.smith@company.org',
            'items': [
                {'product': 'Book', 'price': 24.99, 'qty': 3}
            ],
            'total_amount': '74.97',
            'order_date': '2024-01-15T10:30:00',
            'status': 'shipped'
        }
    ]
    
    print(f"\nProcessing {len(raw_orders)} orders with transformations:")
    
    schema = OrderSchema()
    processed_orders = []
    
    for i, order_data in enumerate(raw_orders):
        print(f"\nOrder {i+1} transformation:")
        print(f"  Original email: {order_data['customer_email']}")
        print(f"  Original amount: {order_data['total_amount']}")
        
        # Load with transformations
        loaded_order = schema.load(order_data)
        processed_orders.append(loaded_order)
        
        print(f"  ✓ Loaded successfully")
        print(f"    Normalized email: {loaded_order['customer_email']}")
        print(f"    Cleaned amount: {loaded_order['total_amount']}")
        print(f"    Item count: {loaded_order['item_count']}")
        print(f"    High value: {loaded_order['is_high_value']}")
        
    
    # both orders are valid, so both must load through the hooks
    assert [(o["customer_email"], str(o["total_amount"]), o["item_count"], o["is_high_value"]) for o in processed_orders] == [
        ("john.doe@example.com", "959.97", 2, True),
        ("jane.smith@company.org", "74.97", 1, False),
    ], processed_orders

    # Serialize back with output transformations
    print(f"\nSerialization with output formatting:")
    
    for i, order in enumerate(processed_orders):
        serialized = schema.dump(order)
        print(f"\nOrder {i+1} serialized:")
        print(f"  Formatted total: {serialized.get('formatted_total')}")
        print(f"  Processed at: {serialized.get('processed_at')}")
        print(f"  Order date: {serialized.get('order_date')}")
        assert serialized["formatted_total"] == ["$959.97", "$74.97"][i], serialized
    
    # Show transformation capabilities
    print(f"\nTransformation hook capabilities:")
    hook_capabilities = [
        "pre_load: Normalize and clean input data before validation",
        "post_load: Add computed fields and business logic after validation",
        "pre_dump: Prepare objects for serialization",
        "post_dump: Format output and add metadata",
        "Data cleaning: Remove unwanted characters, normalize formats",
        "Field computation: Calculate derived values automatically"
    ]
    
    for capability in hook_capabilities:
        print(f"  • {capability}")
    
    print("Data transformation and hooks completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    # Fallback demonstration
    print("Demonstrating transformation concepts:")
    concepts = [
        "Data preprocessing: Clean and normalize input before validation",
        "Hook system: Execute custom logic at different processing stages",
        "Field computation: Automatically calculate derived values", 
        "Output formatting: Control how data appears when serialized",
        "Business logic: Embed domain rules into serialization process"
    ]
    
    for concept in concepts:
        print(f"  - {concept}")
    raise  # re-raise: a broken package must fail this example, not print past it

## Example 4: marshmallow functionality test 4

In [ ]:
from marshmallow import Schema, fields, ValidationError

# Example 4: Nested schemas and complex object relationships
print("Example 4: Nested Schemas and Complex Object Relationships")

try:
    # Define nested schemas for complex data structures
    class AddressSchema(Schema):
        street = fields.Str(required=True)
        city = fields.Str(required=True) 
        state = fields.Str(required=True)
        zip_code = fields.Str(required=True)
        country = fields.Str(load_default='USA')  # `missing=` was removed in marshmallow 4
    
    class CompanySchema(Schema):
        name = fields.Str(required=True)
        industry = fields.Str()
        founded_year = fields.Int()
        headquarters = fields.Nested(AddressSchema)
    
    class EmployeeSchema(Schema):
        employee_id = fields.Int(required=True)
        name = fields.Str(required=True)
        email = fields.Email(required=True)
        department = fields.Str()
        salary = fields.Decimal(places=2)
        hire_date = fields.Date()
        address = fields.Nested(AddressSchema)
        company = fields.Nested(CompanySchema)
    
    print("Nested schemas defined for complex relationships")
    
    # Create complex nested data
    employee_data = {
        'employee_id': 12345,
        'name': 'Sarah Johnson',
        'email': 'sarah.johnson@techcorp.com',
        'department': 'Engineering',
        'salary': '95000.00',
        'hire_date': '2022-03-15',
        'address': {
            'street': '123 Main Street',
            'city': 'San Francisco',
            'state': 'CA',
            'zip_code': '94105',
            'country': 'USA'
        },
        'company': {
            'name': 'TechCorp Inc.',
            'industry': 'Technology',
            'founded_year': 2010,
            'headquarters': {
                'street': '456 Innovation Drive',
                'city': 'Palo Alto',
                'state': 'CA',
                'zip_code': '94301'
            }
        }
    }
    
    print(f"Complex nested employee data created")
    
    # Load nested data
    employee_schema = EmployeeSchema()
    
    loaded_employee = employee_schema.load(employee_data)
    print(f"✓ Successfully loaded employee: {loaded_employee['name']}")
    print(f"  Address: {loaded_employee['address']['city']}, {loaded_employee['address']['state']}")
    print(f"  Company: {loaded_employee['company']['name']}")
    print(f"  HQ Location: {loaded_employee['company']['headquarters']['city']}")
    assert loaded_employee["company"]["headquarters"] == {"street": "456 Innovation Drive", "city": "Palo Alto", "state": "CA",
                                                           "zip_code": "94301", "country": "USA"}, loaded_employee  # country: load_default
    assert (str(loaded_employee["salary"]), loaded_employee["hire_date"].isoformat()) == ("95000.00", "2022-03-15"), loaded_employee
    
    
    # Serialize nested data back
    serialized_employee = employee_schema.dump(loaded_employee)
    print(f"✓ Successfully serialized employee data")
    print(f"  Total keys in output: {len(serialized_employee)}")
    
    # Show nested structure
    nested_fields = []
    for key, value in serialized_employee.items():
        if isinstance(value, dict):
            nested_fields.append(f"{key} ({len(value)} fields)")
    
    print(f"  Nested objects: {', '.join(nested_fields)}")
    assert (len(serialized_employee), nested_fields) == (8, ["address (5 fields)", "company (4 fields)"]), serialized_employee
    assert serialized_employee["hire_date"] == "2022-03-15", serialized_employee
    
    
    # Test list of nested objects
    print(f"\nTesting list of nested objects:")
    
    employees_data = [employee_data]  # List with one employee
    
    # Schema for multiple employees
    employees_schema = EmployeeSchema(many=True)
    
    loaded_employees = employees_schema.load(employees_data)
    print(f"✓ Loaded {len(loaded_employees)} employees from list")
    
    serialized_employees = employees_schema.dump(loaded_employees)
    print(f"✓ Serialized {len(serialized_employees)} employees to list")
    assert loaded_employees == [loaded_employee] and serialized_employees == [serialized_employee], serialized_employees
    
    
    # Partial nested loading
    print(f"\nTesting partial nested data:")
    
    partial_data = {
        'employee_id': 67890,
        'name': 'John Smith',
        'email': 'john@example.com',
        'address': {
            'city': 'New York',
            'state': 'NY'
            # Missing required fields: street, zip_code
        }
    }
    
    try:
        # This should fail due to missing required fields
        result = employee_schema.load(partial_data)
    except ValidationError as e:
        assert e.messages == {'address': {'street': ['Missing data for required field.'],
                                          'zip_code': ['Missing data for required field.']}}, e.messages
        print(f"✓ Correctly rejected incomplete nested data")
    else:
        raise AssertionError("expected ValidationError for the missing nested fields")
    
    # Show nested schema benefits
    print(f"\nNested schema capabilities:")
    nested_capabilities = [
        "Object composition: Model complex real-world relationships",
        "Recursive validation: Validate nested objects with their own rules",
        "Data integrity: Ensure all levels of nesting meet requirements",
        "Flexible depth: Support unlimited nesting levels",
        "Bulk operations: Process lists of complex nested objects",
        "Partial loading: Handle incomplete nested data gracefully"
    ]
    
    for capability in nested_capabilities:
        print(f"  • {capability}")
    
    print("Nested schemas and relationships completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    # Fallback demonstration
    print("Demonstrating nested schema concepts:")
    concepts = [
        "Schema composition: Build complex schemas from simpler ones",
        "Object relationships: Model real-world data relationships",
        "Nested validation: Validate objects at all nesting levels",
        "Data structure preservation: Maintain object hierarchy",
        "Flexible modeling: Support various relationship patterns"
    ]
    
    for concept in concepts:
        print(f"  - {concept}")
    raise  # re-raise: a broken package must fail this example, not print past it

## Example 5: marshmallow functionality test 5

In [ ]:
from marshmallow import Schema, fields
import json

# Example 5: Performance optimization and production patterns
print("Example 5: Performance Optimization and Production Patterns")

try:
    # Define a schema for performance testing
    class APIResponseSchema(Schema):
        status = fields.Str(required=True)
        message = fields.Str()
        data = fields.List(fields.Dict())
        metadata = fields.Dict()
        timestamp = fields.DateTime()
        request_id = fields.Str()
    
    # Create test data for performance testing
    import time
    from datetime import datetime
    
    # Large dataset for performance testing
    large_dataset = []
    for i in range(1000):
        large_dataset.append({
            'status': 'success',
            'message': f'Response {i}',
            'data': [{'id': j, 'value': f'item_{j}'} for j in range(10)],
            'metadata': {'page': i // 100, 'total': 1000},
            'timestamp': datetime.now(),  # Use datetime object, not string
            'request_id': f'req_{i:04d}'
        })
    
    print(f"Created dataset with {len(large_dataset)} records")
    
    # Performance testing
    schema = APIResponseSchema()
    
    # Test single record performance
    single_record = large_dataset[0]
    
    # Serialization performance
    start_time = time.time()
    for _ in range(100):
        serialized = schema.dump(single_record)
    single_serialize_time = time.time() - start_time
    
    print(f"\nSingle record serialization (100x): {single_serialize_time:.4f}s")
    print(f"Average per record: {single_serialize_time * 10:.4f}ms")
    
    # Bulk serialization performance  
    many_schema = APIResponseSchema(many=True)
    
    # Test smaller batch for timing
    test_batch = large_dataset[:100]
    
    start_time = time.time()
    bulk_serialized = many_schema.dump(test_batch)
    bulk_serialize_time = time.time() - start_time
    
    print(f"Bulk serialization (100 records): {bulk_serialize_time:.4f}s")
    print(f"Average per record: {bulk_serialize_time * 10:.4f}ms")
    
    # Deserialization performance
    json_data = json.dumps(test_batch[0], default=str)  # Handle datetime objects in JSON
    dict_data = json.loads(json_data)      # Parse back to dict
    
    start_time = time.time()
    for _ in range(100):
        loaded = schema.load(dict_data)
    deserialize_time = time.time() - start_time
    
    print(f"Deserialization (100x): {deserialize_time:.4f}s")
    print(f"Average per record: {deserialize_time * 10:.4f}ms")
    
    # Memory usage estimation
    import sys
    
    single_dict_size = sys.getsizeof(single_record)
    serialized_size = sys.getsizeof(serialized)
    json_size = sys.getsizeof(json.dumps(serialized, default=str))
    
    print(f"\nMemory usage comparison:")
    print(f"  Original dict: {single_dict_size:,} bytes")
    print(f"  Serialized dict: {serialized_size:,} bytes")
    print(f"  JSON string: {json_size:,} bytes")
    
    # Production optimization patterns
    print(f"\nProduction optimization patterns:")
    
    # 1. Schema reuse
    class OptimizedAPIHandler:
        def __init__(self):
            self.single_schema = APIResponseSchema()
            self.many_schema = APIResponseSchema(many=True)
        
        def serialize_single(self, data):
            return self.single_schema.dump(data)
        
        def serialize_many(self, data_list):
            return self.many_schema.dump(data_list)
        
        def deserialize(self, data):
            return self.single_schema.load(data)
    
    # Test optimized handler
    handler = OptimizedAPIHandler()
    
    start_time = time.time()
    for _ in range(50):
        result = handler.serialize_single(single_record)
    optimized_time = time.time() - start_time
    
    print(f"Optimized handler (50x): {optimized_time:.4f}s")
    print(f"Performance improvement: {single_serialize_time / 2 / optimized_time:.1f}x")
    
    # Show production patterns
    print(f"\nProduction deployment patterns:")
    production_patterns = [
        "Schema reuse: Create schema instances once, reuse for multiple operations",
        "Bulk processing: Use many=True for processing lists of objects",
        "Memory management: Monitor memory usage with large datasets",
        "Error handling: Implement robust error handling for invalid data",
        "Caching: Cache schema instances and validation results where appropriate",
        "Async support: Use with async frameworks for non-blocking I/O",
        "Monitoring: Track serialization performance in production"
    ]
    
    for pattern in production_patterns:
        print(f"  • {pattern}")
    
    # Best practices
    print(f"\nPerformance best practices:")
    best_practices = [
        "Pre-compile schemas: Create schema instances at startup, not per request",
        "Validate early: Validate input data as close to entry point as possible",
        "Batch operations: Process multiple records together when possible",
        "Profile regularly: Monitor serialization performance in production",
        "Optimize hot paths: Focus optimization on frequently used schemas",
        "Memory awareness: Consider memory usage for large data volumes"
    ]
    
    for practice in best_practices:
        print(f"  • {practice}")
    
    print("Performance optimization and patterns completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    # Fallback demonstration
    print("Demonstrating performance concepts:")
    concepts = [
        "Performance testing: Measure serialization/deserialization speed",
        "Schema reuse: Optimize by reusing schema instances",
        "Bulk operations: Efficient processing of multiple records",
        "Memory management: Monitor resource usage in production",
        "Production patterns: Scalable deployment strategies"
    ]
    
    for concept in concepts:
        print(f"  - {concept}")
    raise  # re-raise: a broken package must fail this example, not print past it